In [700]:
#Packages
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
ecoli = pd.read_csv('../ncbi_562_data.csv')
salmonella = pd.read_csv('../ncbi_590_data.csv')

C:\Users\bryan\AppData\Local\Temp\ipykernel_11180\3518567300.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  salmonella = pd.read_csv('../ncbi_590_data.csv')


In [701]:
#USA & >= 2009
ecoli = ecoli[ecoli['is_usa_based'] == 1]
ecoli['collection_year'] = pd.to_numeric(ecoli['collection_year'], errors='coerce')
ecoli = ecoli[ecoli['collection_year'] >= 2009]
ecoli = ecoli[ecoli['month'].notna()]
ecoli['collection_year'] = ecoli['collection_year'].astype(str)
ecoli['collection_year'] = ecoli['collection_year'].str[:4]
ecoli['month'] = ecoli['month'].astype(int)
ecoli['month'] = ecoli['month'].apply(lambda x: f"{x:02d}")
ecoli['year_and_month'] = ecoli['collection_year'] + '-' + ecoli['month']

salmonella = salmonella[salmonella['is_usa_based'] == 1]
salmonella['collection_year'] = pd.to_numeric(salmonella['collection_year'], errors='coerce')
salmonella = salmonella[salmonella['collection_year'] >= 2009]
salmonella = salmonella[salmonella['month'].notna()]
salmonella['collection_year'] = salmonella['collection_year'].astype(str)
salmonella['collection_year'] = salmonella['collection_year'].str[:4]
salmonella['month'] = salmonella['month'].astype(int)
salmonella['month'] = salmonella['month'].apply(lambda x: f"{x:02d}")
salmonella['year_and_month'] = salmonella['collection_year'] + '-' + salmonella['month']

In [702]:
#New Column
ecoli['source_category'] = "Uncategorized"
salmonella['source_category'] = "Uncategorized"

In [703]:
#Condense col
ecoli = ecoli[['source_category','geo_loc_name', 'host', 'isolation_source','year_and_month']]
salmonella = salmonella[['source_category','geo_loc_name', 'host', 'isolation_source','year_and_month']]

In [707]:
def classify_row(host, isolation_source):
    host_str = str(host) if host is not None else ""
    isolation_source_str = str(isolation_source) if isolation_source is not None else ""
    text = f"{host_str} {isolation_source_str}".lower()

    env_terms = ["hospital","river","farm","environment","watershed","sediment",
                 "soil","water","house","surface","floor","air"]
    food_terms = ["meat","carcass","ground","beef","turkey cecum","chicken","egg",
                  "milk","dairy","cheese","fruit","vegetable","apple","lettuce",
                  "spinach","onion","wheat","corn"]

    animals = ["homo sapiens","human","cow","cattle","turkey","chicken","goat",
               "sheep","dog","cat","pig","swine","horse"]

    if any(term in text for term in env_terms):
        return "Environment"
    elif any(term in text for term in food_terms):
        return "Food"
    elif any(term in host_str.lower() for term in animals):
        return "Animal"
    else:
        return "Other/Unknown"

In [708]:
ecoli['source_category'] = ecoli.apply(lambda row: classify_row(row['host'], row['isolation_source']), axis=1)

In [710]:
salmonella['source_category'] = salmonella.apply(lambda row: classify_row(row['host'], row['isolation_source']), axis=1)

In [ ]:
ecoli.to_csv('../ecoli_categories.csv', index=False)
salmonella.to_csv('../salmonella_categories.csv', index=False)

In [699]:
#Replacements Ecoli
ecoli.loc[ecoli['host'] == "Homo sapiens", 'source_category'] = "Human"
ecoli.loc[(ecoli['host'] == "missing") & (ecoli['isolation_source'] == "missing"), 'source_category'] = "Unknown"
ecoli.loc[(ecoli['host'] == "Missing") & (ecoli['isolation_source'] == "Missing"), 'source_category'] = "Unknown"
ecoli.loc[(ecoli['host'] == "Missing") & (ecoli['isolation_source'] == "stool"), 'source_category'] = "Unknown"
ecoli.loc[(ecoli['host'] == "missing") & (ecoli['isolation_source'] == "stool"), 'source_category'] = "Unknown"
ecoli.loc[ecoli['host'] == "Canis lupus familiaris", 'source_category'] = "Dog"
ecoli.loc[ecoli['host'] == "Bos taurus", 'source_category'] = "Cow"
ecoli.loc[(ecoli['host'] == "cattle") & (ecoli['isolation_source'].isin(["feces", "Not Provided"])), 'source_category'] = "Cow"
ecoli.loc[ecoli['host'] == "bovine", 'source_category'] = "Cow"
ecoli.loc[ecoli['isolation_source'].str.contains("beef"), 'source_category'] = "Beef"
ecoli.loc[ecoli['isolation_source'].isin(["flour", "butter"]), 'source_category'] = "Other Food"


ValueError: Cannot mask with non-boolean array containing NA / NaN values

In [ ]:
#Return a random sample of rows
ecoli[ecoli['host'] == "missing"][ecoli['source_category'] == "Uncategorized"]['isolation_source'].value_counts()

C:\Users\bryan\AppData\Local\Temp\ipykernel_11180\4280932587.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ecoli[ecoli['host'] == "missing"][ecoli['source_category'] == "Uncategorized"]['isolation_source'].value_counts()


isolation_source
beef                                                                                          60
Unknown                                                                                       53
parent strain RU1 evolved under selective conditions; Mutation accumulation gen 5900 LB GS    24
parent strain RU1 evolved under selective conditions; LB evolved gen 500 Population GS        24
spinach                                                                                       23
parent strain RU1 evolved under selective conditions; BHI evolved gen 950 Population GS       22
leafy green                                                                                   21
unbleached white flour                                                                        15
reclaimed distribution system                                                                 14
wheat flour                                                                                   14
soy nut butte

In [ ]:
ecoli[ecoli['source_category'] == "Uncategorized"].value_counts('host')

host
missing                                                                   563
porcine                                                                   336
Gallus gallus                                                             326
Felis catus                                                               305
Equus caballus                                                            280
Not Provided                                                              226
cow                                                                       218
Sus scrofa                                                                213
not provided                                                              178
Meleagris gallopavo                                                       139
Gallus gallus domesticus                                                  133
Procyon lotor                                                             123
chicken                                                    